
 Notebook: silver_stores
 Layer: Silver (Transformation)
 Source: bronze.stores

 Description:
 - Cleans and standardizes product data
 - Handles null values and data types
 - Deduplicates records (1 row per product_id)
 - Prepares dataset for downstream joins

 Output:
 - Table: silver.stores

Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Step 1: Read data from Bronze layer

In [0]:
df = spark.table("bronze.stores")

Step 2: Data validation

In [0]:
display(df)
df.count()

Step 3: Basic cleaning

In [0]:
# Remove invalid records (null primary key)
df_clean = df.filter(F.col("store_id").isNotNull())

Step 4: Standardization

In [0]:
# Trim text columns to avoid join issues later

df_clean = df_clean.withColumn("store_name", F.trim(F.col("store_name"))) \
                   .withColumn("city", F.trim(F.col("city"))) \
                   .withColumn("country", F.trim(F.col("country")))

 Step 5: Deduplication

In [0]:
# Keep latest record per store_id

from pyspark.sql.window import Window

window_spec = Window.partitionBy("store_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = df_clean.withColumn("row_num", F.row_number().over(window_spec)) \
                  .filter(F.col("row_num") == 1) \
                  .drop("row_num")

Step 6: Final validation

In [0]:
display(df_dedup)
df_dedup.count()

Step 7: Write to Silver table

In [0]:
df_dedup.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.stores")

Step 8: Verify Silver table

In [0]:
display(spark.table("silver.stores"))